In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")

# The below keys are for langsmith
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [12]:
# DataIngestion - Scrape the data from a website
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(["https://docs.langchain.com/langsmith/profile-configuration"])
documents = loader.load()

print(documents)

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/profile-configuration', 'title': 'Profile configuration - Docs by LangChain', 'description': 'Configure LangSmith SDK credentials and endpoints with a local profile file.', 'language': 'en'}, page_content='Profile configuration - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationProfile configurationGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APILLM GatewayPrivate betaAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusOn this pageMinimum

In [13]:
# Load Data --> Documents --> Chunking --> Text --> Vectors --> Vector Embeddings --> Vector Store

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents=documents)

In [15]:
chunks

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/profile-configuration', 'title': 'Profile configuration - Docs by LangChain', 'description': 'Configure LangSmith SDK credentials and endpoints with a local profile file.', 'language': 'en'}, page_content='Profile configuration - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationProfile configurationGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APILLM GatewayPrivate betaAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusOn this pageMinimum

In [17]:
from langchain_mistralai import MistralAIEmbeddings

embeddings = MistralAIEmbeddings()

In [18]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents=documents, embedding=embeddings)

In [ ]:
vector_store

In [ ]:
# Querying from a vector store

query = "How to override profile values"
result = vector_store.similarity_search(query=query, k=3)
print(result[0].page_content)

Profile configuration - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationProfile configurationGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APILLM GatewayPrivate betaAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusOn this pageMinimum versionsProfile file locationCreate a profile fileSelect a profileManage profiles with the CLIAuthenticate with langsmith auth loginOverride profile valuesUse profiles in codeMount profiles in remote runtimesDockerKubernetesRemote development and CIProfile configurationCop

In [ ]:
# Document Chains
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI

prompt = ChatPromptTemplate.from_template(
  """
    Answer the following question based on the provided context
    <context>
      {context}
    </context>
  """
)

llm = ChatMistralAI(model_name="mistral-small-2506")
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the provided context\n    <context>\n      {context}\n    </context>\n  '), additional_kwargs={})])
| ChatMistralAI(output_version=None, profile={'name': 'Mistral Small 3.2', 'release_date': '2025-06-20', 'last_updated': '2025-06-20', 'open_weights': True, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'atta

In [23]:
from langchain_core.documents import Document

document_chain.invoke({
  "input": "How to override profile values",
  "context": [Document(page_content="Use the LangSmith CLI to create, inspect, switch, and delete profiles without editing the JSON file by hand. To create an API-key profile from an existing API key: export LANGSMITH_API_KEY=<LANGSMITH_API_KEY> langsmith profile create dev \
  --workspace-id <WORKSPACE_ID> \
  --set-current You can also pass the key and endpoint as flags. Prefer environment variables on shared machines, because shell history may record command flags.")]
})

"Based on the provided context, here's how to create an API-key profile using the LangSmith CLI:\n\n1. **Using environment variables** (recommended for shared machines):\n   ```bash\n   export LANGSMITH_API_KEY=<YOUR_API_KEY>\n   langsmith profile create dev --workspace-id <WORKSPACE_ID> --set-current\n   ```\n\n2. **Using command-line flags** (alternative):\n   ```bash\n   langsmith profile create dev --workspace-id <WORKSPACE_ID> --api-key <YOUR_API_KEY> --set-current\n   ```\n\nKey points:\n- Replace `<YOUR_API_KEY>` with your actual LangSmith API key\n- Replace `<WORKSPACE_ID>` with your workspace ID\n- The `--set-current` flag makes this profile the active one\n- Environment variables are preferred for security on shared machines"

In [24]:
# Retrievers

# Input --> Retrievers --> Vectorstore --> Result

retriever = vector_store.as_retriever()

In [25]:
retriever

VectorStoreRetriever(tags=['FAISS', 'MistralAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026053703A40>, search_kwargs={})

In [27]:
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [28]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'MistralAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026053703A40>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based on the provided context\n    <context>\n      {context}\n    </context>\n  '), additional_kwargs={})])
 